# Huấn luyện Random Forest 4 lớp với 18 features, tránh leakage

Notebook này giữ đúng hướng triển khai runtime của `ZTA_controller.py`: dùng 18 features gồm 15 raw features + 3 group features.

Điểm khác biệt quan trọng so với notebook cũ:
- Không tính group features trên toàn bộ dataset trước khi chia tập.
- Chia train/val/test bằng `GroupShuffleSplit` để giảm leakage giữa các burst traffic gần nhau.
- Chỉ tính lại 3 group features sau khi từng split đã tách riêng.

Lưu ý: hiện dataset chưa có `capture_id`/`session_id`, nên notebook tạm dùng group gần đúng theo `source_file + ip_src + cửa sổ thời gian 10 giây`. Nếu sau này có `capture_id`, hãy thay group đó bằng `capture_id` để đánh giá còn sạch hơn.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

BASE_DIR = os.getcwd()
TARGET_NAMES = ['BENIGN', 'DDoS', 'DoS', 'Port Scan']
RANDOM_STATE = 42
TIME_BUCKET_SECONDS = 10


## Bước 1: Nạp dữ liệu 4 lớp và giữ source metadata

In [2]:
files = {
    'benign': ('benign.csv', 0),
    'ddos': ('ddos.csv', 1),
    'dos': ('dos.csv', 2),
    'portscan': ('portscan.csv', 3),
}

benign_dfs = []
attack_dfs = {1: [], 2: [], 3: []}

for source_name, (fname, attack_label) in files.items():
    path = os.path.join(BASE_DIR, fname)
    if not os.path.exists(path):
        print(f'Skipping missing file: {fname}')
        continue

    df = pd.read_csv(path)
    df['source_file'] = source_name

    if source_name == 'benign':
        df['label'] = 0
        benign_dfs.append(df)
        continue

    attack_ips = {'10.0.0.7', '10.0.0.8'}
    if attack_label == 1:
        attack_ips.add('10.0.0.6')

    is_attack = df['ip_src'].astype(str).isin(attack_ips)

    df_attack = df[is_attack].copy()
    df_attack['label'] = attack_label
    attack_dfs[attack_label].append(df_attack)

    df_background = df[~is_attack].copy()
    df_background['label'] = 0
    benign_dfs.append(df_background)

df_all = pd.concat(
    [pd.concat(benign_dfs, ignore_index=True)] +
    [pd.concat(attack_dfs[label], ignore_index=True) for label in sorted(attack_dfs) if attack_dfs[label]],
    ignore_index=True
)

print(f'Tong so mau: {len(df_all):,}')
print(df_all['label'].value_counts().sort_index())
df_all.head(3)

Tong so mau: 896,509
label
0    507662
1    165343
2     83369
3    140135
Name: count, dtype: int64


,timestamp,datapath_id,flow_id,ip_src,tp_src,ip_dst,tp_dst,ip_proto,icmp_code,icmp_type,flow_duration_sec,flow_duration_nsec,idle_timeout,hard_timeout,flags,packet_count,byte_count,packet_count_per_second,packet_count_per_nsecond,byte_count_per_second,byte_count_per_nsecond,label,source_file
0,1.781810e+09,4,10.0.0.5010.0.0.701,10.0.0.5,0,10.0.0.7,0,1,0,0,0,114000000,20,100,0,0,0,0.0,0.000000e+00,0.0,0.000000,0,benign
1,1.781810e+09,4,10.0.0.7010.0.0.501,10.0.0.7,0,10.0.0.5,0,1,0,8,0,121000000,20,100,0,0,0,0.0,0.000000e+00,0.0,0.000000,0,benign
2,1.781810e+09,1,10.0.0.18010.0.0.3525426,10.0.0.1,80,10.0.0.3,52542,6,-1,-1,0,111000000,20,100,0,4,1526,0.0,3.603604e-08,0.0,0.000014,0,benign


## Bước 2: Tạo group để split sạch hơn

Ở đây group được định nghĩa theo:
- `source_file`
- `ip_src`
- cửa sổ thời gian 10 giây

Mục tiêu là giữ các burst traffic gần nhau nằm cùng một tập, giảm việc một kiểu luồng gần như giống hệt xuất hiện ở cả train và test.

In [3]:
df_all = df_all.copy()
df_all['timestamp_bucket'] = (df_all['timestamp'] // TIME_BUCKET_SECONDS).astype('int64')
df_all['split_group'] = (
    df_all['source_file'].astype(str) + '|' +
    df_all['ip_src'].astype(str) + '|' +
    df_all['timestamp_bucket'].astype(str)
)

print('So group:', df_all['split_group'].nunique())
print(df_all[['source_file', 'ip_src', 'timestamp_bucket', 'split_group']].head())

So group: 7591
  source_file    ip_src  timestamp_bucket                split_group
0      benign  10.0.0.5         178180991  benign|10.0.0.5|178180991
1      benign  10.0.0.7         178180991  benign|10.0.0.7|178180991
2      benign  10.0.0.1         178180991  benign|10.0.0.1|178180991
3      benign  10.0.0.1         178180991  benign|10.0.0.1|178180991
4      benign  10.0.0.1         178180991  benign|10.0.0.1|178180991


## Bước 3: Group-aware split train/test

In [4]:
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss_test.split(df_all, y=df_all['label'], groups=df_all['split_group']))

df_train = df_all.iloc[train_idx].reset_index(drop=True)
df_test = df_all.iloc[test_idx].reset_index(drop=True)

print(f'Train: {len(df_train):,} samples | groups={df_train["split_group"].nunique():,}')
print(f'Test : {len(df_test):,} samples | groups={df_test["split_group"].nunique():,}')

for name, frame in [('Train', df_train), ('Test', df_test)]:
    print(f'\n{name} label distribution:')
    print(frame['label'].value_counts(normalize=True).sort_index().round(4))

Train: 625,573 samples | groups=6,072
Test : 270,936 samples | groups=1,519

Train label distribution:
label
0    0.5628
1    0.2146
2    0.0889
3    0.1337
Name: proportion, dtype: float64

Test label distribution:
label
0    0.5742
1    0.1148
2    0.1025
3    0.2085
Name: proportion, dtype: float64


## Bước 4: Tính lại 3 group features bên trong từng split

Đây là chỗ quan trọng nhất để tránh leakage.

In [5]:
RAW_FEATURES = [
    'tp_dst', 'ip_proto', 'icmp_code', 'icmp_type',
    'flow_duration_sec', 'flow_duration_nsec',
    'idle_timeout', 'hard_timeout', 'flags',
    'packet_count', 'byte_count',
    'packet_count_per_second', 'packet_count_per_nsecond',
    'byte_count_per_second', 'byte_count_per_nsecond',
]

GROUP_FEATURES = ['flow_count_src', 'packet_rate_src', 'unique_dst_ports_src']
ALL_FEATURES = RAW_FEATURES + GROUP_FEATURES

def recalculate_group_features(df):
    df = df.copy()
    df['round_time'] = (df['timestamp'] // TIME_BUCKET_SECONDS).astype('int64')
    grouped = df.groupby(['round_time', 'ip_src'])

    flow_count = grouped.size().to_dict()
    packet_rate = grouped['packet_count_per_second'].sum().to_dict()
    unique_ports = grouped['tp_dst'].nunique().to_dict()

    keys = list(zip(df['round_time'], df['ip_src']))
    df['flow_count_src'] = [flow_count.get(k, 0) for k in keys]
    df['packet_rate_src'] = [packet_rate.get(k, 0.0) for k in keys]
    df['unique_dst_ports_src'] = [unique_ports.get(k, 0) for k in keys]

    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
    return df.drop(columns=['round_time'])

df_train_feat = recalculate_group_features(df_train)
df_test_feat = recalculate_group_features(df_test)

X_train = df_train_feat[ALL_FEATURES].astype('float64')
y_train = df_train_feat['label'].astype('int64')
X_test = df_test_feat[ALL_FEATURES].astype('float64')
y_test = df_test_feat['label'].astype('int64')

print('Feature count:', X_train.shape[1])
print('Features:', list(X_train.columns))

Feature count: 18
Features: ['tp_dst', 'ip_proto', 'icmp_code', 'icmp_type', 'flow_duration_sec', 'flow_duration_nsec', 'idle_timeout', 'hard_timeout', 'flags', 'packet_count', 'byte_count', 'packet_count_per_second', 'packet_count_per_nsecond', 'byte_count_per_second', 'byte_count_per_nsecond', 'flow_count_src', 'packet_rate_src', 'unique_dst_ports_src']


## Bước 5: Huấn luyện Random Forest

In [6]:
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

clf.fit(X_train, y_train)
print('Da huan luyen xong.')

Da huan luyen xong.


## Bước 6: Đánh giá Test

In [7]:
def evaluate_split(name, X, y):
    y_pred = clf.predict(X)
    acc = accuracy_score(y, y_pred)
    print(f'\n===== {name} =====')
    print(f'Accuracy: {acc * 100:.4f}%')
    print('Confusion matrix:')
    print(confusion_matrix(y, y_pred))
    print('\nClassification report:')
    print(classification_report(y, y_pred, labels=range(4), target_names=TARGET_NAMES, zero_division=0))
    return acc

test_acc = evaluate_split('Test', X_test, y_test)


===== Test =====
Accuracy: 96.7239%
Confusion matrix:
[[153472     20   2081      0]
 [    12  25090   5996      0]
 [     0      0  27393    382]
 [   385      0      0  56105]]

Classification report:
              precision    recall  f1-score   support

      BENIGN       1.00      0.99      0.99    155573
        DDoS       1.00      0.81      0.89     31098
         DoS       0.77      0.99      0.87     27775
   Port Scan       0.99      0.99      0.99     56490

    accuracy                           0.97    270936
   macro avg       0.94      0.94      0.94    270936
weighted avg       0.97      0.97      0.97    270936



## Bước 7: Kiểm tra overfitting theo train/test gap

In [8]:
train_pred = clf.predict(X_train)
train_acc = accuracy_score(y_train, train_pred)
gap = train_acc - test_acc

print(f'Train Accuracy: {train_acc * 100:.4f}%')
print(f'Test  Accuracy: {test_acc * 100:.4f}%')
print(f'Gap           : {gap * 100:.4f}%')

if gap < 0.02:
    print('Gap < 2%: khong co dau hieu overfit ro rang theo train/test gap.')
else:
    print('Gap >= 2%: can regularization manh hon hoac split sach hon.')

Train Accuracy: 99.2802%
Test  Accuracy: 96.7239%
Gap           : 2.5562%
Gap >= 2%: can regularization manh hon hoac split sach hon.


## Bước 8: Group-aware cross-validation

In [9]:
df_cv = recalculate_group_features(df_all)
X_cv = df_cv[ALL_FEATURES].astype('float64')
y_cv = df_cv['label'].astype('int64')
groups_cv = df_cv['split_group']

group_kfold = GroupKFold(n_splits=5)
cv_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=15,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

cv_scores = cross_val_score(
    cv_model,
    X_cv,
    y_cv,
    cv=group_kfold,
    groups=groups_cv,
    scoring='accuracy',
    n_jobs=-1,
)

print('Group CV Scores:', cv_scores)
print(f'Group CV Mean : {cv_scores.mean() * 100:.4f}% +/- {cv_scores.std() * 100:.4f}%')

Group CV Scores: [0.98390983 0.9871669  0.94464089 0.84734693 0.94896849]
Group CV Mean : 94.2407% +/- 5.0617%


## Bước 9: Feature importance và lưu mô hình

In [ ]:
importances = pd.DataFrame({
    'feature': clf.feature_names_in_,
    'importance': clf.feature_importances_,
}).sort_values('importance', ascending=False)

print(importances.to_string(index=False))

model_path = os.path.join(BASE_DIR, 'random_forest_multiclass_zta.pkl')
joblib.dump(clf, model_path)
print(f'\nDa luu model tai: {model_path}')

                 feature  importance
    unique_dst_ports_src    0.237084
          flow_count_src    0.226357
         packet_rate_src    0.152140
                  tp_dst    0.150920
              byte_count    0.067620
            packet_count    0.054193
   byte_count_per_second    0.030075
 packet_count_per_second    0.021192
                ip_proto    0.020505
       flow_duration_sec    0.015553
               icmp_code    0.009479
               icmp_type    0.008115
      flow_duration_nsec    0.005210
  byte_count_per_nsecond    0.001402
packet_count_per_nsecond    0.000154
            idle_timeout    0.000000
            hard_timeout    0.000000
                   flags    0.000000

Da luu model tai: /home/vuxtrung/KLTN_ver2/Source Code/Model/random_forest_multiclass_zta.pkl
